# Splice Site Prediction Using Foundation Model Embeddings

**Pipeline Overview:**
1. Extract embeddings from foundation models (offline, one-time)
2. Train lightweight 3-class classifiers on embeddings
3. Evaluate using 5-fold cross-validation
4. Compare results across models and window sizes

**Expected duration:** ~3-4 hours total

**Models:** HyenaDNA, DNABert, NucleotideTransformer

**Window sizes:** 300, 600, 1000, 2000, 5000, 10000 bp

**Nucleotide Transformer limits:**
- NT v1 500M runs up to window 6000, so it supports 5000 and skips only window 10000 in this notebook
- NT v2 runs up to window 12000, so it supports all configured windows here

## Setup: Import Libraries and Configuration

In [1]:
import os
import sys
import json
import importlib
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Force-disable flash/triton attention kernels for compatibility
os.environ['FLASH_ATTENTION_DISABLE'] = '1'
os.environ['USE_FLASH_ATTENTION'] = '0'
os.environ['TRITON_DISABLE_LINE_INFO'] = '1'

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

import config
import models
import splicing_embed_extract
import splicing_train
importlib.reload(config)
importlib.reload(models)
importlib.reload(splicing_embed_extract)
importlib.reload(splicing_train)

from config import *
from splicing_embed_extract import EmbeddingExtractor
from splicing_classifier import SpliceSiteClassifier
from splicing_dataset import EmbeddingDataset, create_embedding_dataloader
from splicing_metrics import MetricsComputer
from splicing_train import SpliceClassifierTrainer

print(f"✓ Imports successful (reloaded latest src modules)")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Window sizes: {WINDOW_SIZES}")

✓ Imports successful (reloaded latest src modules)
Device: NVIDIA GeForce RTX 5080
Project directory: d:\Splice_FMs_seq_lengths
Window sizes: [300, 600, 1000, 2000, 5000, 10000]


## Configuration Check

In [2]:
# Verify data files exist
print("Data files available:")
for ws in WINDOW_SIZES:
    gencode_dir = RAW_DATA_DIR / f'gencode{ws}.csv'
    gtex_dir = GTEX_DATA_DIR / f'gtex{ws}.csv'
    print(f"  Window {ws}:")
    print(f"    GENCODE: {gencode_dir.exists()}")
    print(f"    GTEx: {gtex_dir.exists()}")

print(f"\nOutput directories:")
print(f"  Embeddings: {EMBEDDINGS_DIR}")
print(f"  Results: {SPLICING_RESULTS_DIR}")
print(f"  TensorBoard: {SPLICING_TENSORBOARD_DIR}")

print(f"\nEmbedding extraction config:")
print(f"  Method: {EMBEDDING_CONFIG['method']}")
print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']} (GPU memory safety)")
print(f"  Adaptive batch sizes by window:")
for ws, bs in EMBEDDING_CONFIG.get('batch_size_by_window', {}).items():
    print(f"    Window {ws:5d} bp: batch_size = {bs}")

print(f"\nModel/window support:")
for model_name, model_cfg in MODELS_CONFIG.items():
    for model_id in model_cfg['model_ids']:
        supported = [ws for ws in WINDOW_SIZES if is_model_window_supported(model_name, model_id, ws)]
        skipped = [ws for ws in WINDOW_SIZES if not is_model_window_supported(model_name, model_id, ws)]
        print(f"  {model_id.split('/')[-1]}")
        print(f"    Supported: {supported}")
        if skipped:
            print(f"    Skipped:   {skipped}")

Data files available:
  Window 300:
    GENCODE: True
    GTEx: True
  Window 600:
    GENCODE: True
    GTEx: True
  Window 1000:
    GENCODE: True
    GTEx: True
  Window 2000:
    GENCODE: True
    GTEx: True
  Window 5000:
    GENCODE: True
    GTEx: True
  Window 10000:
    GENCODE: True
    GTEx: True

Output directories:
  Embeddings: d:\Splice_FMs_seq_lengths\data\embeddings
  Results: d:\Splice_FMs_seq_lengths\results\classifiers
  TensorBoard: d:\Splice_FMs_seq_lengths\logs\splicing_classifiers

Embedding extraction config:
  Method: center
  Max length: 10000
  Use FP16: True (GPU memory safety)
  Adaptive batch sizes by window:
    Window   300 bp: batch_size = 256
    Window   600 bp: batch_size = 128
    Window  1000 bp: batch_size = 64
    Window  2000 bp: batch_size = 32
    Window  5000 bp: batch_size = 8
    Window 10000 bp: batch_size = 4

Model/window support:
  hyenadna-small-32k-seqlen-hf
    Supported: [300, 600, 1000, 2000, 5000, 10000]
  hyenadna-medium-160k-se

## PHASE 1: Extract Embeddings (Optional - Run Once)

In [3]:
# Skip this cell if embeddings are already extracted
EXTRACT_EMBEDDINGS = True  # Set to True to extract embeddings

if EXTRACT_EMBEDDINGS:
    print("Extracting embeddings from foundation models...")
    print("Configuration:")
    print(f"  Method: {EMBEDDING_CONFIG['method']}")
    print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
    print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']}")
    print(f"  Device: {EMBEDDING_CONFIG['device']}")
    print(f"\nAdaptive batch sizes:")
    for ws, bs in EMBEDDING_CONFIG.get('batch_size_by_window', {}).items():
        print(f"  Window {ws:5d} bp: {bs:3d}")
    print("\nUnsupported NT combinations are skipped automatically:")
    print("  - NT v1 500M: skip window 10000 only")
    print("  - NT v2: no skip for the configured windows [300, 600, 1000, 2000, 5000, 10000]")
    print(f"\nThis will take ~2-3 hours on RTX 5080\n")
    
    extractor = EmbeddingExtractor(device='cuda')
    stats = extractor.extract_all()
    
    print(f"\n✓ Embedding extraction completed")
    print(f"  Extracted: {stats['total_succeeded']}")
    print(f"  Skipped existing: {stats.get('total_skipped_existing', 0)}")
    print(f"  Skipped unsupported: {stats.get('total_skipped_unsupported', 0)}")
    print(f"  Failed: {stats['total_failed']}")
else:
    print("Skipping embedding extraction.")
    print("To extract embeddings, set EXTRACT_EMBEDDINGS = True and run this cell.")

2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - Configured safe attention backend: flash_sdp=False, mem_efficient_sdp=False, math_sdp=True
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - 
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - EMBEDDING EXTRACTION - START
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - ================================================================================
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - Window sizes: [300, 600, 1000, 2000, 5000, 10000]
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - Models: ['HyenaDNA', 'DNABert', 'NucleotideTransformer']
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - Device: cuda
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - Output: d:\Splice_FMs_seq_lengths\data\embeddings
2026-03-08 15:50:34,823 - splicing_embed_extract - INFO - 
################################################################################
2026-03-08 15:50:34,823 

Extracting embeddings from foundation models...
Configuration:
  Method: center
  Max length: 10000
  Use FP16: True
  Device: cuda

Adaptive batch sizes:
  Window   300 bp: 256
  Window   600 bp: 128
  Window  1000 bp:  64
  Window  2000 bp:  32
  Window  5000 bp:   8
  Window 10000 bp:   4

Unsupported NT combinations are skipped automatically:
  - NT v1 500M: skip window 10000 only
  - NT v2: no skip for the configured windows [300, 600, 1000, 2000, 5000, 10000]

This will take ~2-3 hours on RTX 5080



2026-03-08 15:50:35,477 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 15:50:35,477 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-08 15:50:35,516 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/LongSafari/hyenadna-small-32k-seqlen-hf/8fe770c78eb13fe33bf81501612faeddf4d6f331/config.json "HTTP/1.1 200 OK"
2026-03-08 15:50:35,790 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf/resolve/main/configuration_hyena.py "HTTP/1.1 307 Temporary Redirect"
2026-03-08 15:50:35,826 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/LongSafari/hyenadna-small-32k-seqlen-hf/8fe770c78eb13fe33bf81501612faeddf4d6f331/configuration

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

HyenaDNAModel LOAD REPORT from: LongSafari/hyenadna-small-32k-seqlen-hf
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-08 15:50:38,783 - models - INFO - Successfully loaded HyenaDNA model: LongSafari/hyenadna-small-32k-seqlen-hf
2026-03-08 15:50:38,783 - splicing_embed_extract - INFO - ✓ Model loaded
2026-03-08 15:50:38,783 - splicing_embed_extract - INFO - Loading GENCODE data for window size 5000...
2026-03-08 15:50:38,783 - data_preparation - INFO - Loading gencode data: d:\Splice_FMs_seq_lengths\gencode_multi_seq_length\gencode5000.csv
2026-03-08 15:50:56,038 - data_preparation - INFO - Loaded 744617 samples from gencode5000
2026-03-08 15:50:56,471 - data_preparation - INFO - Split sizes - Train: 610515, Val: 107738, Test: 26364
2026-03-08 15:50:56,486 - splicing_embed_extract - INFO - ✓ GENCODE loade

KeyboardInterrupt: 

## PHASE 2: List Available Embeddings

In [ ]:
# Check which embeddings are available for CV training
available_combinations = []
unsupported_combinations = []
missing_supported_combinations = []

for window_size in WINDOW_SIZES:
    for model_name in MODELS_CONFIG.keys():
        for model_id in MODELS_CONFIG[model_name]['model_ids']:
            combo_name = f"{model_name}_{model_id}"
            
            if not is_model_window_supported(model_name, model_id, window_size):
                unsupported_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                    'reason': get_model_window_skip_reason(model_name, model_id, window_size),
                })
                continue

            embed_dir = EMBEDDINGS_DIR / str(window_size) / combo_name
            trainval_file = embed_dir / "trainval_embeddings.pt"
            test_file = embed_dir / "test_embeddings.pt"
            gtex_file = embed_dir / "gtex_test_embeddings.pt"
            
            if trainval_file.exists() and test_file.exists() and gtex_file.exists():
                available_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                    'combo_name': combo_name,
                    'embed_dir': embed_dir
                })
            else:
                missing_supported_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                })

print(f"Available embeddings for CV training: {len(available_combinations)}")
for combo in available_combinations:
    print(f"  ✓ Window {combo['window_size']:5d} | {combo['combo_name']:30s}")

print(f"\nUnsupported combinations skipped by design: {len(unsupported_combinations)}")
for combo in unsupported_combinations:
    print(f"  ↷ Window {combo['window_size']:5d} | {combo['model_id']} | {combo['reason']}")

if missing_supported_combinations:
    print(f"\nSupported combinations still missing embeddings: {len(missing_supported_combinations)}")
    for combo in missing_supported_combinations:
        print(f"  - Window {combo['window_size']:5d} | {combo['model_id']}")

if len(available_combinations) == 0:
    print("\n⚠ No supported trainval/test embeddings found. Please run embedding extraction first.")

Available embeddings for CV training: 27
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   300 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   600 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
  ✓

## PHASE 3: Train Classifiers with 5-Fold CV

In [ ]:
# Train classifiers for each supported combination
all_results = {}
training_start_time = datetime.now()

# Skip training if result file already exists
SKIP_IF_RESULTS_EXIST = True

for idx, combo in enumerate(available_combinations):
    if not is_model_window_supported(combo['model_name'], combo['model_id'], combo['window_size']):
        print(
            f"\n↷ Skipped unsupported combination: {combo['combo_name']} "
            f"(Window {combo['window_size']})"
        )
        continue

    print(f"\n{'='*80}")
    print(f"[{idx+1}/{len(available_combinations)}] Training: {combo['combo_name']} (Window {combo['window_size']})")
    print(f"{'='*80}")
    
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / "results.json"

    # Skip if already trained, but still load into all_results for downstream summary
    if SKIP_IF_RESULTS_EXIST and results_file.exists():
        try:
            with open(results_file, 'r') as f:
                existing_results = json.load(f)
            all_results[experiment_name] = existing_results
            print(f"↷ Skipped (already trained): {experiment_name}")
            print(f"  Loaded existing results: {results_file}")
            continue
        except Exception as e:
            print(f"⚠ Found existing results but failed to load ({e}), retraining...")
    
    # Load train+val embeddings prepared for CV
    trainval_file = combo['embed_dir'] / "trainval_embeddings.pt"
    trainval_data = torch.load(trainval_file)
    
    all_embeddings = trainval_data['embeddings']
    all_labels = trainval_data['labels']
    
    embedding_dim = all_embeddings.shape[1]
    
    # Train with CV
    trainer = SpliceClassifierTrainer(
        embedding_dim=embedding_dim,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        results_dir=str(SPLICING_RESULTS_DIR)
    )
    
    results = trainer.train_with_cv(
        all_embeddings.numpy(),
        all_labels.numpy(),
        experiment_name=experiment_name,
        num_folds=5,
        num_epochs=50,
        batch_size=256,
        learning_rate=0.0001,
        weight_decay=0.001,
        early_stopping_patience=5,
        seed=42,
        deterministic=False
    )
    
    all_results[experiment_name] = results
    
    print(f"✓ {experiment_name} training completed")

elapsed_hours = (datetime.now() - training_start_time).total_seconds() / 3600
print(f"\n{'='*80}")
print(f"Total training time: {elapsed_hours:.1f} hours")
print(f"Total experiments available in all_results: {len(all_results)}")
print(f"{'='*80}")

2026-03-07 17:03:06,057 - splicing_train - INFO - 



[1/26] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 300)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-small-32k-seqlen-hf_w300\results.json

[2/26] Training: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf (Window 300)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-medium-160k-seqlen-hf_w300\results.json

[3/26] Training: DNABert_zhihan1996/DNABERT-2-117M (Window 300)
↷ Skipped (already trained): DNABert_zhihan1996/DNABERT-2-117M_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\DNABert_zhihan1996\DNABERT-2-117M_w300\results.json

[4/26] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref (Window 300)
↷ Skipped (already trained)

2026-03-07 17:03:06,057 - splicing_train - INFO - TRAINING: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w2000
2026-03-07 17:03:06,058 - splicing_train - INFO - ================================================================================
2026-03-07 17:03:06,058 - splicing_train - INFO - Embeddings shape: (718253, 256)
2026-03-07 17:03:06,059 - splicing_train - INFO - Labels shape: (718253,)
2026-03-07 17:03:06,060 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-07 17:03:06,061 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-07 17:03:06,116 - splicing_train - INFO - 
################################################################################
2026-03-07 17:03:06,117 - splicing_train - INFO - FOLD 1/5
2026-03-07 17:03:06,118 - splicing_train - INFO - ################################################################################
2026-03-07 17:03:06,191 - splicing_train - INFO - Train: 574602, Val: 143651
2026-03-07 17:

✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w2000 training completed

[20/26] Training: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf (Window 2000)


2026-03-07 17:16:43,556 - splicing_train - INFO - 
2026-03-07 17:16:43,557 - splicing_train - INFO - TRAINING: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w2000
2026-03-07 17:16:43,557 - splicing_train - INFO - ================================================================================
2026-03-07 17:16:43,559 - splicing_train - INFO - Embeddings shape: (718253, 256)
2026-03-07 17:16:43,559 - splicing_train - INFO - Labels shape: (718253,)
2026-03-07 17:16:43,560 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-07 17:16:43,562 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-07 17:16:43,623 - splicing_train - INFO - 
################################################################################
2026-03-07 17:16:43,624 - splicing_train - INFO - FOLD 1/5
2026-03-07 17:16:43,624 - splicing_train - INFO - ################################################################################
2026-03-07 17:16:43,707 - splicing_tra

...

✓ DNABert_zhihan1996/DNABERT-2-117M_w2000 training completed

[22/26] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species (Window 2000)


2026-03-07 18:20:05,752 - splicing_train - INFO - 
2026-03-07 18:20:05,752 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w2000
2026-03-07 18:20:05,754 - splicing_train - INFO - ================================================================================
2026-03-07 18:20:05,754 - splicing_train - INFO - Embeddings shape: (718253, 512)
2026-03-07 18:20:05,754 - splicing_train - INFO - Labels shape: (718253,)
2026-03-07 18:20:05,755 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-07 18:20:05,756 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-07 18:20:05,818 - splicing_train - INFO - 
################################################################################
2026-03-07 18:20:05,819 - splicing_train - INFO - FOLD 1/5
2026-03-07 18:20:05,819 - splicing_train - INFO - ################################################################################
2026-03-07

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w2000 training completed

[23/26] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species (Window 2000)


2026-03-07 18:52:26,822 - splicing_train - INFO - 
2026-03-07 18:52:26,822 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w2000
2026-03-07 18:52:26,823 - splicing_train - INFO - ================================================================================
2026-03-07 18:52:26,823 - splicing_train - INFO - Embeddings shape: (718253, 768)
2026-03-07 18:52:26,823 - splicing_train - INFO - Labels shape: (718253,)
2026-03-07 18:52:26,824 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-07 18:52:26,825 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-07 18:52:26,885 - splicing_train - INFO - 
################################################################################
2026-03-07 18:52:26,885 - splicing_train - INFO - FOLD 1/5
2026-03-07 18:52:26,886 - splicing_train - INFO - ################################################################################
2026-03-07

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w2000 training completed

[24/26] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 10000)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w10000
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-small-32k-seqlen-hf_w10000\results.json

[25/26] Training: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf (Window 10000)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w10000
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-medium-160k-seqlen-hf_w10000\results.json

[26/26] Training: DNABert_zhihan1996/DNABERT-2-117M (Window 10000)
↷ Skipped (already trained): DNABert_zhihan1996/DNABERT-2-117M_w10000
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\DNABert_zhihan1996\DNABERT-2-117M_w10000\results.json

To

## PHASE 4: Results Comparison and Analysis

In [ ]:
# Summarize results + export aggregate files
summary_data = []

summary_columns = [
    'Experiment',
    'Window',
    'Accuracy',
    'Balanced Acc',
    'F1 (weighted)',
    'F1 (macro)',
    'MCC',
    'ROC-AUC',
]

for exp_name, results in all_results.items():
    avg_metrics = results.get('averaged_metrics', {})
    if not isinstance(avg_metrics, dict):
        avg_metrics = {}
    
    # Parse window size from experiment name suffix: ..._w300
    window_size = None
    if '_w' in exp_name:
        try:
            window_size = int(exp_name.rsplit('_w', 1)[-1])
        except ValueError:
            window_size = None

    summary_data.append({
        'Experiment': exp_name,
        'Window': window_size,
        'Accuracy': avg_metrics.get('accuracy_mean', 0),
        'Balanced Acc': avg_metrics.get('balanced_accuracy_mean', 0),
        'F1 (weighted)': avg_metrics.get('f1_weighted_mean', 0),
        'F1 (macro)': avg_metrics.get('f1_macro_mean', 0),
        'MCC': avg_metrics.get('mcc_mean', 0),
        'ROC-AUC': avg_metrics.get('roc_auc_weighted_mean', 0),
    })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    for col in summary_columns:
        if col not in summary_df.columns:
            summary_df[col] = np.nan
    summary_df = summary_df[summary_columns]
    summary_df = summary_df.sort_values('F1 (weighted)', ascending=False, na_position='last')
else:
    summary_df = pd.DataFrame(columns=summary_columns)

print("\n" + "="*100)
print("RESULTS SUMMARY (Sorted by F1-Weighted)")
print("="*100)
if summary_df.empty:
    print("No results available in all_results. Run PHASE 3 first or check loaded results files.")
else:
    print(summary_df.to_string(index=False))
print("="*100)

# Export aggregate summaries
summary_dir = SPLICING_RESULTS_DIR / 'summaries'
summary_dir.mkdir(parents=True, exist_ok=True)

all_csv = summary_dir / 'all_experiments_summary.csv'
all_json = summary_dir / 'all_experiments_summary.json'
summary_df.to_csv(all_csv, index=False)
summary_df.to_json(all_json, orient='records', indent=2)

print(f"\nSaved aggregate summaries:")
print(f"  - {all_csv}")
print(f"  - {all_json}")

# Export one file per window
windows_exported = []
if not summary_df.empty and 'Window' in summary_df.columns:
    unique_windows = [w for w in summary_df['Window'].dropna().unique()]
    for ws in sorted(unique_windows):
        ws_int = int(ws)
        ws_df = summary_df[summary_df['Window'] == ws].copy()
        ws_df = ws_df.sort_values('F1 (weighted)', ascending=False, na_position='last')
        ws_csv = summary_dir / f'summary_w{ws_int}.csv'
        ws_json = summary_dir / f'summary_w{ws_int}.json'
        ws_df.to_csv(ws_csv, index=False)
        ws_df.to_json(ws_json, orient='records', indent=2)
        windows_exported.append((ws_csv, ws_json))

if windows_exported:
    print("\nSaved per-window summaries:")
    for ws_csv, ws_json in windows_exported:
        print(f"  - {ws_csv}")
        print(f"  - {ws_json}")


RESULTS SUMMARY (Sorted by F1-Weighted)
                                                                          Experiment  Window  Accuracy  Balanced Acc  F1 (weighted)  F1 (macro)      MCC  ROC-AUC
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w600     600  0.919324      0.919305       0.919226    0.919169 0.879048 0.985398
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w300     300  0.909771      0.909743       0.909545    0.909466 0.864782 0.981873
       NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w1000    1000  0.890775      0.890805       0.890460    0.890473 0.836253 0.975664
NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w2000    2000  0.834656      0.834478       0.833890    0.833593 0.752712 0.945051
NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w2000    2000  0.826341      0.826151       0.825472    0.825166 0.740

## PHASE 5: Best Model Evaluation on Test Set

In [ ]:
# Inference with best-fold checkpoint for each experiment on GENCODE test and GTEx test embeddings
print("\nRunning inference using best-fold checkpoints...\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INFER_BATCH_SIZE = 1024
inference_results = {}

def _to_serializable(obj):
    if isinstance(obj, dict):
        return {k: _to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_serializable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def _resolve_best_checkpoint(results_dir, cv_results):
    best_fold_info = cv_results.get('best_fold', {})

    checkpoint_path = best_fold_info.get('checkpoint_path', None)
    checkpoint_name = best_fold_info.get('checkpoint', None)
    fold_number = best_fold_info.get('fold_number', None)

    # Fallback for old results format without best_fold
    if checkpoint_name is None:
        fold_entries = cv_results.get('per_fold_results', {})
        if not fold_entries:
            return None, None, None

        best_key = None
        best_mcc = -1e9
        for fold_key, fold_val in fold_entries.items():
            mcc = fold_val.get('best_mcc', -1e9)
            if mcc > best_mcc:
                best_mcc = mcc
                best_key = fold_key

        if best_key is None:
            return None, None, None

        checkpoint_name = fold_entries[best_key].get('best_checkpoint', None)
        if checkpoint_name is None:
            # If old results do not contain checkpoint name, infer from fold index
            try:
                fold_idx = int(best_key.replace('fold_', ''))
                checkpoint_name = f"best_fold{fold_idx + 1}.pt"
            except Exception:
                checkpoint_name = None

        try:
            fold_number = int(best_key.replace('fold_', '')) + 1
        except Exception:
            fold_number = None

    checkpoint_path_obj = None
    if checkpoint_path is not None:
        checkpoint_path_obj = Path(checkpoint_path)
        if not checkpoint_path_obj.exists():
            checkpoint_path_obj = None

    if checkpoint_path_obj is None and checkpoint_name is not None:
        checkpoint_path_obj = results_dir / 'checkpoints' / checkpoint_name

    if checkpoint_path_obj is None or not checkpoint_path_obj.exists():
        return None, checkpoint_name, fold_number

    return checkpoint_path_obj, checkpoint_name, fold_number

def _run_inference_on_embedding_file(checkpoint_payload, embedding_file, batch_size=1024):
    data = torch.load(embedding_file, map_location='cpu')
    embeddings = data['embeddings']
    labels = data['labels']

    if not isinstance(embeddings, torch.Tensor):
        embeddings = torch.tensor(embeddings, dtype=torch.float32)
    else:
        embeddings = embeddings.float()

    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels, dtype=torch.long)
    else:
        labels = labels.long()

    model_state = checkpoint_payload.get('model_state_dict', checkpoint_payload)
    embedding_dim = int(checkpoint_payload.get('embedding_dim', embeddings.shape[1]))
    num_classes = int(checkpoint_payload.get('num_classes', 3))

    model = SpliceSiteClassifier(
        embedding_dim=embedding_dim,
        num_classes=num_classes,
        hidden_dims=[512, 256],
        dropout_rate=0.3,
    ).to(device)
    model.load_state_dict(model_state)
    model.eval()

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for start_idx in range(0, len(embeddings), batch_size):
            end_idx = min(start_idx + batch_size, len(embeddings))
            batch_x = embeddings[start_idx:end_idx].to(device)
            logits = model(batch_x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())

    y_true = labels.cpu().numpy()
    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_prob = torch.cat(all_probs, dim=0).numpy()

    metrics, cm = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)
    return {
        'num_samples': int(len(y_true)),
        'metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in metrics.items()},
        'confusion_matrix': cm.tolist(),
    }

for combo in available_combinations:
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / 'results.json'

    if not results_file.exists():
        print(f"⚠ Missing results.json: {results_file}")
        continue

    with open(results_file, 'r') as f:
        cv_results = json.load(f)

    checkpoint_path, checkpoint_name, fold_number = _resolve_best_checkpoint(results_dir, cv_results)
    if checkpoint_path is None:
        print(f"⚠ Missing best-fold checkpoint for {experiment_name}")
        continue

    test_embed_file = combo['embed_dir'] / 'test_embeddings.pt'
    gtex_embed_file = combo['embed_dir'] / 'gtex_test_embeddings.pt'

    if not test_embed_file.exists() or not gtex_embed_file.exists():
        print(f"⚠ Missing test embedding files for {experiment_name}")
        continue

    checkpoint_payload = torch.load(checkpoint_path, map_location='cpu')

    gencode_test_eval = _run_inference_on_embedding_file(
        checkpoint_payload=checkpoint_payload,
        embedding_file=test_embed_file,
        batch_size=INFER_BATCH_SIZE,
    )
    gtex_test_eval = _run_inference_on_embedding_file(
        checkpoint_payload=checkpoint_payload,
        embedding_file=gtex_embed_file,
        batch_size=INFER_BATCH_SIZE,
    )

    inference_results[experiment_name] = {
        'window_size': combo['window_size'],
        'model_name': combo['model_name'],
        'model_id': combo['model_id'],
        'best_fold_number': fold_number,
        'checkpoint': checkpoint_name,
        'checkpoint_path': str(checkpoint_path),
        'gencode_test': gencode_test_eval,
        'gtex_test': gtex_test_eval,
    }

    print(
        f"✓ {experiment_name} | fold={fold_number} | "
        f"GENCODE F1={gencode_test_eval['metrics'].get('f1_weighted', 0):.4f}, "
        f"GTEx F1={gtex_test_eval['metrics'].get('f1_weighted', 0):.4f}"
    )

print(f"\n✓ Inference completed for {len(inference_results)} experiments")

# Save inference summary
inference_summary_dir = SPLICING_RESULTS_DIR / 'summaries'
inference_summary_dir.mkdir(parents=True, exist_ok=True)
inference_results_file = inference_summary_dir / 'best_fold_inference_results.json'
with open(inference_results_file, 'w') as f:
    json.dump(_to_serializable(inference_results), f, indent=2)

print(f"Saved inference results: {inference_results_file}")


Running inference using best-fold checkpoints...

✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300 | fold=3 | GENCODE F1=0.7732, GTEx F1=0.7877
✓ HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w300 | fold=5 | GENCODE F1=0.8008, GTEx F1=0.8127
✓ DNABert_zhihan1996/DNABERT-2-117M_w300 | fold=5 | GENCODE F1=0.5801, GTEx F1=0.5775
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w300 | fold=3 | GENCODE F1=0.9057, GTEx F1=0.9076
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w300 | fold=2 | GENCODE F1=0.7895, GTEx F1=0.8012
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w300 | fold=2 | GENCODE F1=0.7779, GTEx F1=0.7836
✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w600 | fold=5 | GENCODE F1=0.7818, GTEx F1=0.7960
✓ HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w600 | fold=5 | GENCODE F1=0.8183, GTEx F1=0.8317
✓ DNABert_zhihan1996/DNABERT-2-117M_w600 | fold=1 | GENCODE F1=0.5452

## Summary

In [ ]:
print("\n" + "="*80)
print("PIPELINE SUMMARY")
print("="*80)

print(f"\n✓ Embedding extraction: {len(available_combinations)} combinations")
print(f"✓ Classifier training: 5-fold CV with early stopping")
print(f"✓ Results saved:")
print(f"  - JSON files: {SPLICING_RESULTS_DIR}")
print(f"  - TensorBoard logs: {SPLICING_TENSORBOARD_DIR}")

print(f"\nBest performing model (F1-weighted):")
if 'summary_df' in globals() and not summary_df.empty:
    best_row = summary_df.iloc[0]
    print(f"  {best_row['Experiment']}")
    print(f"    F1-weighted: {best_row['F1 (weighted)']:.4f}")
    print(f"    Accuracy: {best_row['Accuracy']:.4f}")
    print(f"    MCC: {best_row['MCC']:.4f}")
else:
    print("  No available result row to report.")

print(f"\nView TensorBoard results:")
print(f"  tensorboard --logdir {SPLICING_TENSORBOARD_DIR}")
print(f"\n" + "="*80)


PIPELINE SUMMARY

✓ Embedding extraction: 26 combinations
✓ Classifier training: 5-fold CV with early stopping
✓ Results saved:
  - JSON files: d:\Splice_FMs_seq_lengths\results\classifiers
  - TensorBoard logs: d:\Splice_FMs_seq_lengths\logs\splicing_classifiers

Best performing model (F1-weighted):
  NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w600
    F1-weighted: 0.9192
    Accuracy: 0.9193
    MCC: 0.8790

View TensorBoard results:
  tensorboard --logdir d:\Splice_FMs_seq_lengths\logs\splicing_classifiers

